In [1]:
!pip install -q sentence-transformers faiss-cpu openai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [3]:
documents = [
    "NovaTech Solutions was founded in 2024.",
    "NovaTech Solutions has three departments: Data Analytics, Artificial Intelligence, and Marketing.",
    "The company's main internal project is called Project Orion.",
    "Project Orion was officially launched in March 2026.",
    "The project manager of Project Orion has employee ID NT204.",
    "NovaTech Solutions headquarters is located in Lucknow, India.",
    "The company uses Python, SQL, Power BI, and machine learning for its data analytics work.",
    "NovaTech Solutions has 50 employees.",
    "The company's internal customer-support system is called NovaHelp.",
    "NovaHelp was introduced in January 2026."
]

print("Total documents:", len(documents))

Total documents: 10


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

embeddings = np.array(
    embeddings,
    dtype="float32"
)

print("Embedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (10, 384)
Data type: float32


In [5]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors in FAISS:", index.ntotal)
print("Vector dimension:", index.d)

Total vectors in FAISS: 10
Vector dimension: 384


In [24]:
def semantic_search(query, top_k=3):
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding, dtype="float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for i, idx in enumerate(indices[0]):
        results.append({
            "document": documents[idx],
            "distance": distances[0][i]
        })

    return results

In [7]:
query = "When was Project Orion launched?"

results = semantic_search(query, top_k=3)

for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("Document:", result["document"])
    print("Distance:", result["distance"])


Result 1
Document: Project Orion was officially launched in March 2026.
Distance: 0.30209643

Result 2
Document: The company's main internal project is called Project Orion.
Distance: 0.43852916

Result 3
Document: The project manager of Project Orion has employee ID NT204.
Distance: 0.71896034


In [10]:
from google.colab import userdata
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

print("OpenAI connected successfully!")

OpenAI connected successfully!


In [11]:
def generate_with_rag(query):
    results = semantic_search(query, top_k=3)

    context = "\n\n".join(
        [result["document"] for result in results]
    )

    prompt = f"""
You are a helpful assistant.
Answer the question ONLY using the information
provided in the context.

If the answer is not present in the context, say:
"I don't have enough information in the provided context."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [13]:
!pip install -q transformers sentencepiece

In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Free local model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Free local model loaded successfully!


In [16]:
def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [17]:
def generate_with_rag(query):
    results = semantic_search(query, top_k=3)

    context = "\n\n".join(
        [result["document"] for result in results]
    )

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    return generate_answer(prompt)

In [18]:
def generate_without_rag(query):
    prompt = f"""
Answer the following question:

Question:
{query}

Answer:
"""

    return generate_answer(prompt)

In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Tokenizer and model loaded successfully!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Tokenizer and model loaded successfully!


In [21]:
def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [23]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

rag_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
rag_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("RAG generation model loaded!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


RAG generation model loaded!


In [26]:
def generate_answer(prompt):
    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = rag_model.generate(
        **inputs,
        max_new_tokens=100
    )

    return rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [27]:
def generate_with_rag(query):
    results = semantic_search(query, top_k=3)

    context = "\n\n".join(
        [result["document"] for result in results]
    )

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    return generate_answer(prompt)

In [28]:
def generate_without_rag(query):
    prompt = f"""
Answer the following question:

Question:
{query}

Answer:
"""

    return generate_answer(prompt)

In [29]:
query = "When was Project Orion launched?"

print("ANSWER WITH RAG:")
print(generate_with_rag(query))

print("\nANSWER WITHOUT RAG:")
print(generate_without_rag(query))

ANSWER WITH RAG:
March 2026

ANSWER WITHOUT RAG:
October 1, 2017


In [30]:
questions = [
    "When was NovaTech Solutions founded?",
    "How many departments does NovaTech Solutions have?",
    "What is the name of the company's main internal project?",
    "Where is NovaTech Solutions headquarters located?",
    "What is the name of the company's customer-support system?"
]
for i, question in enumerate(questions, 1):
    print(f"\nQuestion {i}: {question}")

    print("With RAG:")
    print(generate_with_rag(question))

    print("Without RAG:")
    print(generate_without_rag(question))

    print("-" * 60)


Question 1: When was NovaTech Solutions founded?
With RAG:
2024
Without RAG:
June 1, 1899
------------------------------------------------------------

Question 2: How many departments does NovaTech Solutions have?
With RAG:
three
Without RAG:
ten
------------------------------------------------------------

Question 3: What is the name of the company's main internal project?
With RAG:
Project Orion
Without RAG:
adolescence
------------------------------------------------------------

Question 4: Where is NovaTech Solutions headquarters located?
With RAG:
Lucknow
Without RAG:
New York City
------------------------------------------------------------

Question 5: What is the name of the company's customer-support system?
With RAG:
NovaHelp
Without RAG:
e-commerce
------------------------------------------------------------


In [31]:
comparison = []

for question in questions:
    rag_answer = generate_with_rag(question)
    no_rag_answer = generate_without_rag(question)

    comparison.append({
        "Question": question,
        "With RAG": rag_answer,
        "Without RAG": no_rag_answer
    })

comparison_df = pd.DataFrame(comparison)

comparison_df

,Question,With RAG,Without RAG
0,When was NovaTech Solutions founded?,2024,"June 1, 1899"
1,How many departments does NovaTech Solutions h...,three,ten
2,What is the name of the company's main interna...,Project Orion,adolescence
3,Where is NovaTech Solutions headquarters located?,Lucknow,New York City
4,What is the name of the company's customer-sup...,NovaHelp,e-commerce


In [32]:
def trace_rag(query):
    results = semantic_search(query, top_k=3)

    print("QUESTION:")
    print(query)

    print("\nRETRIEVED DOCUMENTS:")
    for i, result in enumerate(results, 1):
        print(f"{i}. {result['document']}")
        print(f"   Distance: {result['distance']}")

    print("\nRAG ANSWER:")
    print(generate_with_rag(query))

In [33]:
trace_rag("What is the employee ID of the Project Orion manager?")

QUESTION:
What is the employee ID of the Project Orion manager?

RETRIEVED DOCUMENTS:
1. The project manager of Project Orion has employee ID NT204.
   Distance: 0.1699155867099762
2. The company's main internal project is called Project Orion.
   Distance: 0.6948948502540588
3. Project Orion was officially launched in March 2026.
   Distance: 0.9906687140464783

RAG ANSWER:
NT204


In [35]:
trace_rag("When was NovaHelp introduced?")

QUESTION:
When was NovaHelp introduced?

RETRIEVED DOCUMENTS:
1. NovaHelp was introduced in January 2026.
   Distance: 0.3356148600578308
2. The company's internal customer-support system is called NovaHelp.
   Distance: 0.8639273643493652
3. NovaTech Solutions was founded in 2024.
   Distance: 0.8683767318725586

RAG ANSWER:
January 2026


RAG Architecture

User Query
    -->
Query Embedding
    -->
FAISS Vector Search
    -->
Top 3 Relevant Documents
    -->
Context + Query
    -->
FLAN-T5 Generation Model
    -->
Grounded Answer


Without RAG

User Query
    -->
FLAN-T5 Generation Model
    -->
Answer Without Context